# Fine-Tuning BART on Scientific Papers (PDF + Figures)

This notebook walks through the full pipeline for fine-tuning
**facebook/bart-large-cnn** on scientific papers that come as PDFs and
may contain figures.

### Pipeline overview

1. **PDF processing** — extract text *and* raster images from each paper
2. **Figure captioning** — run extracted images through **BLIP** (a PyTorch
   vision-language model) to produce text descriptions, so BART can
   "see" the figures
3. **Target extraction** — pull the abstract from each paper as the
   ground-truth summary (or supply your own via a JSON manifest)
4. **Tokenisation & batching** — standard BART tokeniser with padding /
   truncation
5. **Training** — AdamW + cosine schedule, gradient accumulation, mixed
   precision, ROUGE evaluation, best-model checkpointing
6. **Inference** — summarise new papers with the fine-tuned model

## 0 — Install dependencies

Run once if you haven't already.

In [ ]:
# !pip install -r requirements.txt

## 1 — Imports & configuration

In [ ]:
import torch
from pathlib import Path

from paper_dataset import DatasetConfig, PaperDataset
from trainer import TrainConfig, BartTrainer

print(f"PyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"MPS available:  {hasattr(torch.backends, 'mps') and torch.backends.mps.is_available()}")

In [ ]:
# ---------------------------------------------------------------------------
# Paths — edit these to point at your data
# ---------------------------------------------------------------------------

# Directory containing your scientific-paper PDFs
PAPERS_DIR = Path("papers")            # <-- change me

# (Optional) JSON manifest mapping each PDF to a human-written summary.
# If None, the abstract extracted from each PDF is used as the target.
# Format:  [{"pdf": "paper_001.pdf", "summary": "..."}, ...]
MANIFEST_PATH = None                   # <-- or Path("manifest.json")

# Where to save checkpoints & TensorBoard logs
OUTPUT_DIR = Path("checkpoints")

## 2 — Dataset configuration

`DatasetConfig` controls PDF processing, BLIP captioning, and BART
tokenisation.  Adjust these if your papers are unusually long or
figure-heavy.

In [ ]:
ds_config = DatasetConfig(
    # BART tokeniser settings
    bart_model_name="facebook/bart-large-cnn",
    max_source_tokens=1024,
    max_target_tokens=256,

    # BLIP figure-captioning model
    blip_model_name="Salesforce/blip-image-captioning-base",
    max_caption_tokens=48,

    # Image filtering
    min_image_width=100,
    min_image_height=100,
    max_images_per_paper=10,
)

print(f"Caption device: {ds_config.caption_device}")

## 3 — Build the dataset

This step will:
- scan every PDF in `PAPERS_DIR`
- extract text + images
- caption each figure with BLIP
- extract or look up the target summary
- tokenise everything for BART

It can take a while on first run because the BLIP model needs to be
downloaded (~1 GB) and each figure is captioned individually.

In [ ]:
if MANIFEST_PATH is not None:
    dataset = PaperDataset.from_manifest(
        manifest_path=MANIFEST_PATH,
        papers_dir=PAPERS_DIR,
        config=ds_config,
        use_captioner=True,
    )
else:
    dataset = PaperDataset.from_directory(
        papers_dir=PAPERS_DIR,
        config=ds_config,
        use_captioner=True,
    )

print(f"\nDataset size: {len(dataset)} papers")

### 3.1 — Inspect a sample

In [ ]:
sample = dataset[0]
print("Keys:", list(sample.keys()))
print(f"input_ids shape:      {sample['input_ids'].shape}")
print(f"attention_mask shape: {sample['attention_mask'].shape}")
print(f"labels shape:         {sample['labels'].shape}")

# Decode and preview
src_preview = dataset.tokenizer.decode(
    sample["input_ids"][:80], skip_special_tokens=True
)
lbl = sample["labels"].clone()
lbl[lbl == -100] = dataset.tokenizer.pad_token_id
tgt_preview = dataset.tokenizer.decode(lbl[:60], skip_special_tokens=True)

print(f"\n--- Source (first 80 tokens) ---\n{src_preview}")
print(f"\n--- Target (first 60 tokens) ---\n{tgt_preview}")

## 4 — Training configuration

In [ ]:
train_config = TrainConfig(
    model_name="facebook/bart-large-cnn",

    # Optimiser
    learning_rate=3e-5,
    weight_decay=0.01,

    # Schedule
    warmup_ratio=0.06,
    num_epochs=5,

    # Batching — effective batch size = batch_size * gradient_accumulation_steps
    batch_size=2,
    gradient_accumulation_steps=8,

    # Mixed precision (turn off if unstable on MPS)
    use_amp=True,

    # Evaluation generation
    eval_max_tokens=256,
    eval_num_beams=4,

    # Output
    output_dir=str(OUTPUT_DIR),
    save_every_n_epochs=1,
    log_every_n_steps=5,
    use_tensorboard=True,

    # Data split
    val_fraction=0.1,
)

print(f"Device: {train_config.device}")
print(f"AMP dtype: {train_config.amp_dtype}")
print(f"Effective batch size: {train_config.batch_size * train_config.gradient_accumulation_steps}")

## 5 — Train

In [ ]:
trainer = BartTrainer(dataset, config=train_config)

In [ ]:
trainer.train()

## 6 — Inference with the fine-tuned model

Use the trainer's built-in helper, or load the saved checkpoint
directly.

In [ ]:
# --- Option A: use the trainer object (model is already in memory) ---

test_text = dataset.records[0]["source"][:3000]  # first paper, truncated
summary = trainer.generate_summary(test_text)
print("=" * 72)
print("GENERATED SUMMARY")
print("=" * 72)
print(summary)

In [ ]:
# --- Option B: load the best checkpoint from disk ---

from transformers import BartForConditionalGeneration, BartTokenizer

ckpt_path = OUTPUT_DIR / "best_model"

tokenizer = BartTokenizer.from_pretrained(str(ckpt_path))
model = BartForConditionalGeneration.from_pretrained(str(ckpt_path))
model.eval()

inputs = tokenizer(
    test_text, max_length=1024, truncation=True, return_tensors="pt"
)
with torch.inference_mode():
    ids = model.generate(
        inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_length=256,
        num_beams=4,
        no_repeat_ngram_size=3,
        early_stopping=True,
    )

print(tokenizer.decode(ids[0], skip_special_tokens=True))

## 7 — Summarise a brand-new PDF end-to-end

Combines the fine-tuned BART model with the hierarchical summariser
from `summarizer.py` to handle an arbitrarily long paper.

In [ ]:
from summarizer import BookSummarizer, SummarizerConfig

# Point at the fine-tuned checkpoint instead of the base model
infer_cfg = SummarizerConfig(
    model_name=str(OUTPUT_DIR / "best_model"),
)

summarizer = BookSummarizer(infer_cfg)

# Summarise any PDF — results are saved to an output directory
NEW_PDF = Path("path/to/new_paper.pdf")  # <-- change me

final = summarizer.summarize_pdf(NEW_PDF, output_dir="new_paper_summary")
print("\n" + "=" * 72)
print(final)